In [52]:
from dash import Dash, html, dcc
from dash.dependencies import Input, Output
import plotly.express as px
import seaborn as sns

In [53]:
app = Dash(
    external_scripts =["https://unpkg.com/@tailwindcss/browser@4"]
)

tips = sns.load_dataset("tips")

fig_bar = px.bar(
    tips,
    x="day",
    y="tip",
    color="day",
    title="Gorjeta média por dia da semana",
    labels={"day": "Dia da semana", "tip": "Gorjeta"}
)



In [54]:
app.layout = html.Div(
    children= [ 
        html.H1("Homura did nothing wrong",
            className = "text-blue-600 text-4xl"),
        html.Br(),
        html.Div(
            children=[
                html.Label("Filtrar por fumantes:"),
                # dcc.Checklist → permite múltiplas seleções
                dcc.Checklist(
                    id="checklist-fumante",
                    options=[
                        {"label": "Fumantes", "value": "Yes"},
                        {"label": "Não fumantes", "value": "No"}
                    ],
                    # value é uma LISTA (pois pode selecionar múltiplos)
                    value=["Yes", "No"],
                    inline=True
                )
            ],
            style={"marginBottom": "20px"}
        ),

        html.Div(
            children=[
                html.Label("Tamanho mínimo da mesa (número de pessoas):"),
                # dcc.Slider → componente de seleção numérica contínua/discreta
                dcc.Slider(
                    id="slider-tamanho",
                    min=int(tips["size"].min()),
                    max=int(tips["size"].max()),
                    step=1,
                    value=2,
                    # As chaves do dicionário 'marks' precisam ser tipos simples (int/str).
                    # Aqui convertemos explicitamente para int para evitar erro de numpy.int64.
                    marks={int(i): str(int(i)) for i in sorted(tips["size"].unique())}
                )
            ],
            style={"marginBottom": "40px"}
        ),

        html.Div(
            children=[
                html.P("Os componentes acima ainda não estão conectados a nenhum gráfico."),
                html.P("No próximo exemplo, vamos criar callbacks para torná-los interativos.")
            ]
        ),
        html.Div(
            children=[
                html.H1("Dashboard de Gorjetas - Callback Simples"),

                html.Label("Selecione o dia da semana:"),
                dcc.Dropdown(
                    id="dropdown-dia-callback",
                    options=[{"label": dia, "value": dia} for dia in tips["day"].unique()],
                    value="Sun",
                    clearable=False
                ),

                dcc.Graph(id="grafico-filtrado")
            ],
            style={"padding": "20px"}
        )
    ],
               
)

@app.callback(
    # "Atualize a propriedade 'figure' do componente 'grafico-filtrado'"
    Output("grafico-filtrado", "figure"),  # componente de saída: o gráfico
    # "Observe mudanças no valor selecionado do dropdown"
    Input("dropdown-dia-callback", "value")  # componente de entrada: valor do dropdown
)
def atualizar_grafico(dia_selecionado):
    """Filtra o DataFrame pelo dia selecionado e retorna uma figura Plotly."""
    df_filtrado = tips[tips["day"] == dia_selecionado]

    fig = px.scatter(
        df_filtrado,
        x="total_bill",
        y="tip",
        color="sex",
        title=f"Relação Conta x Gorjeta - Dia: {dia_selecionado}"
    )
    return fig



app.run(debug=True, port = 8080)